# **Predicting Trading Signals**

## **Introduction**

Through this project we will develop and evaluate supervised machine learning models to predict ’buy’ and ’sell’ signals for stocks, using manually computed technical indicators such as MACD, RSI, and Bollinger Bands. We will use the prepared data from project 2. 

### **Load the prepared dataset**

In [2]:
# Import libraries
import pandas as pd
import numpy as np

In [3]:
# Import the prepared dataset from Project 2

stock_data = pd.read_csv(
    "../Project_2_DataCleaning_Preparation/stock_prepared.csv")

In [4]:
stock_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 20860334 entries, 0 to 20860333
Data columns (total 29 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   ticker                        str    
 1   date                          str    
 2   open                          float64
 3   high                          float64
 4   low                           float64
 5   close                         float64
 6   volume                        int64  
 7   price_change                  float64
 8   daily_return                  float64
 9   close_lag_1                   float64
 10  ma_5                          float64
 11  ma_20                         float64
 12  volatility_20                 float64
 13  avg_volume_20                 float64
 14  exchange_NASDAQ               int64  
 15  exchange_NYSE                 int64  
 16  sector_BASIC INDUSTRIES       int64  
 17  sector_CAPITAL GOODS          int64  
 18  sector_CONSUMER DURABLES      i

In [5]:
# Convert date to date time

stock_data["date"] = pd.to_datetime(stock_data["date"])

In [6]:
stock_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 20860334 entries, 0 to 20860333
Data columns (total 29 columns):
 #   Column                        Dtype         
---  ------                        -----         
 0   ticker                        str           
 1   date                          datetime64[us]
 2   open                          float64       
 3   high                          float64       
 4   low                           float64       
 5   close                         float64       
 6   volume                        int64         
 7   price_change                  float64       
 8   daily_return                  float64       
 9   close_lag_1                   float64       
 10  ma_5                          float64       
 11  ma_20                         float64       
 12  volatility_20                 float64       
 13  avg_volume_20                 float64       
 14  exchange_NASDAQ               int64         
 15  exchange_NYSE                 int64      

## **Part 1 — Create Technical Indicators**

#### **1.1 Calculate MACD**

In [7]:
# Order the data by ticker an date

stock_data = stock_data.sort_values(["ticker", "date"])

In [ ]:
# calculate the 12-day window Exponential Moving Average (EMA) of close

stock_data["EMA12"] = (
    stock_data.groupby("ticker")["close"]
    .transform(lambda x: x.ewm(span=12, adjust=False).mean()))

In [ ]:
# Calculate 26-day window EMA

stock_data["EMA26"] = (
    stock_data.groupby("ticker")["close"]
    .transform(lambda x: x.ewm(span=26, adjust=False).mean()))

In [10]:
# Calculate MACD line

stock_data["MACD"] = (
    stock_data["EMA12"] - stock_data["EMA26"])

In [ ]:
# Calculate 9-day EMA of MACD (Signal line)

stock_data["Signal_Line"] = (
    stock_data.groupby("ticker")["MACD"]
    .transform(lambda x: x.ewm(span=9, adjust=False).mean()))

In [12]:
stock_data[
    ["ticker", "date", "close", "EMA12", "EMA26", "MACD", "Signal_Line"]
].head(30)

,ticker,date,close,EMA12,EMA26,MACD,Signal_Line
0,A,1999-12-17,32.859444,32.859444,32.859444,0.000000,0.000000
1,A,1999-12-20,33.530045,32.962613,32.909118,0.053495,0.010699
2,A,1999-12-21,33.351215,33.022398,32.941866,0.080532,0.024666
3,A,1999-12-22,34.021816,33.176155,33.021862,0.154293,0.050591
4,A,1999-12-23,35.586552,33.546985,33.211839,0.335146,0.107502
5,A,1999-12-27,37.777184,34.197785,33.550013,0.647772,0.215556
6,A,1999-12-28,43.991417,35.704497,34.323450,1.381047,0.448654
7,A,1999-12-29,51.502148,38.134905,35.595946,2.538959,0.866715
8,A,1999-12-30,56.688126,40.989247,37.158330,3.830917,1.459556
9,A,1999-12-31,55.302216,43.191242,38.502321,4.688921,2.105429


### **Create and Compare the MACD line with its signal line.**

In [13]:
# Get previous MACD and Signal Line

stock_data["prev_MACD"] = stock_data.groupby("ticker")["MACD"].shift(1)
stock_data["prev_Signal"] = stock_data.groupby("ticker")["Signal_Line"].shift(1)

In [14]:
# Create Buy, Sell, and Hold conditions

# Define crossover conditions
buy_condition = (
    (stock_data["prev_MACD"] < stock_data["prev_Signal"]) &
    (stock_data["MACD"] > stock_data["Signal_Line"]))

sell_condition = (
    (stock_data["prev_MACD"] > stock_data["prev_Signal"]) &
    (stock_data["MACD"] < stock_data["Signal_Line"]))

In [ ]:
# Create MACD signal

stock_data["MACD_Signal"] = "Hold"

stock_data.loc[buy_condition, "MACD_Signal"] = "Buy"
stock_data.loc[sell_condition, "MACD_Signal"] = "Sell"

In [ ]:
# Check the results

stock_data[
    ["ticker", "date", "MACD", "Signal_Line",
     "prev_MACD", "prev_Signal", "MACD_Signal"]
].head(30)

,ticker,date,MACD,Signal_Line,prev_MACD,prev_Signal,MACD_Signal
0,A,1999-12-17,0.000000,0.000000,NaN,NaN,Hold
1,A,1999-12-20,0.053495,0.010699,0.000000,0.000000,Hold
2,A,1999-12-21,0.080532,0.024666,0.053495,0.010699,Hold
3,A,1999-12-22,0.154293,0.050591,0.080532,0.024666,Hold
4,A,1999-12-23,0.335146,0.107502,0.154293,0.050591,Hold
5,A,1999-12-27,0.647772,0.215556,0.335146,0.107502,Hold
6,A,1999-12-28,1.381047,0.448654,0.647772,0.215556,Hold
7,A,1999-12-29,2.538959,0.866715,1.381047,0.448654,Hold
8,A,1999-12-30,3.830917,1.459556,2.538959,0.866715,Hold
9,A,1999-12-31,4.688921,2.105429,3.830917,1.459556,Hold


In [17]:
stock_data["MACD_Signal"].value_counts()

MACD_Signal
Hold    19123625
Buy       868779
Sell      867930
Name: count, dtype: int64

#### **Insights**:

We calculated the MACD (Moving Average Convergence Divergence) separately for each stock using its closing price. First, we calculated the 12-day EMA (EMA12) and 26-day EMA (EMA26). We then calculated the MACD line as the difference between these two moving averages

Next, we calculated the Signal Line as the 9-day EMA of the MACD. We then compared the current and previous MACD and Signal Line values to identify crossovers. A Buy signal was generated when MACD crossed above the Signal Line, a Sell signal when MACD crossed below it, and Hold when neither crossover occurred.

These signals describe MACD-based trading conditions according to the rules defined for this project and are not guarantees of future stock-price movements.

### **1.2 Calculate RSI**

In [ ]:
# Calculate price change

stock_data["RSI_change"] = stock_data.groupby("ticker")["close"].diff()

In [ ]:
# Separate gains and losses

stock_data["gain"] = stock_data["RSI_change"].clip(lower=0)
stock_data["loss"] = -stock_data["RSI_change"].clip(upper=0)

In [ ]:
# Calculate 14-day exponential average gain

stock_data["avg_gain"] = (
    stock_data.groupby("ticker")["gain"]
    .transform(lambda x: x.ewm(span=14, adjust=False).mean()))

In [ ]:
# Calculate 14-day exponential average loss

stock_data["avg_loss"] = (
    stock_data.groupby("ticker")["loss"]
    .transform(lambda x: x.ewm(span=14, adjust=False).mean()))

In [ ]:
# Calculate Relative Strength

stock_data["RS"] = stock_data["avg_gain"] / stock_data["avg_loss"]

In [ ]:
# Calculate RSI

stock_data["RSI"] = 100 - (100 / (1 + stock_data["RS"]))

In [24]:
# Create RSI signal
stock_data["RSI_Signal"] = "Hold"

stock_data.loc[stock_data["RSI"] < 30, "RSI_Signal"] = "Buy"
stock_data.loc[stock_data["RSI"] > 70, "RSI_Signal"] = "Sell"

In [25]:
# Check the results

stock_data[
    ["ticker", "date", "close", "RSI_change",
     "gain", "loss", "avg_gain", "avg_loss",
     "RS", "RSI", "RSI_Signal"]
].head(30)

,ticker,date,close,RSI_change,gain,loss,avg_gain,avg_loss,RS,RSI,RSI_Signal
0,A,1999-12-17,32.859444,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hold
1,A,1999-12-20,33.530045,0.670601,0.670601,-0.000000,0.670601,-0.000000,-inf,100.000000,Sell
2,A,1999-12-21,33.351215,-0.178829,0.000000,0.178829,0.581187,0.023844,24.374688,96.059065,Sell
3,A,1999-12-22,34.021816,0.670601,0.670601,-0.000000,0.593109,0.020665,28.701556,96.633173,Sell
4,A,1999-12-23,35.586552,1.564735,1.564735,-0.000000,0.722659,0.017909,40.350815,97.581668,Sell
5,A,1999-12-27,37.777184,2.190632,2.190632,-0.000000,0.918389,0.015521,59.168869,98.338011,Sell
6,A,1999-12-28,43.991417,6.214233,6.214233,-0.000000,1.624502,0.013452,120.763204,99.178734,Sell
7,A,1999-12-29,51.502148,7.510731,7.510731,-0.000000,2.409332,0.011658,206.661255,99.518447,Sell
8,A,1999-12-30,56.688126,5.185978,5.185978,-0.000000,2.779552,0.010104,275.096478,99.637808,Sell
9,A,1999-12-31,55.302216,-1.385910,0.000000,1.385910,2.408945,0.193545,12.446450,92.563093,Sell


In [26]:
# Check the signal distribution

stock_data["RSI_Signal"].value_counts()

RSI_Signal
Hold    16051721
Sell     2696316
Buy      2112297
Name: count, dtype: int64

#### **Insights:**

We calculated the RSI (Relative Strength Index) separately for each stock using changes in its closing price. First, we calculated the daily price changes and separated them into gains and losses. We then calculated the 14-day exponential average gain and average loss.

Next, we calculated the Relative Strength (RS) and used it to calculate the RSI, which ranges from 0 to 100.

Finally, we created the RSI signals according to the project rules. An RSI below 30 generated a Buy signal, an RSI above 70 generated a Sell signal, and values between 30 and 70 generated a Hold signal.

These signals represent overbought and oversold conditions according to the RSI rules used in this project and do not guarantee future stock-price movements.